# Experimental PSA Benchmark — CsA Ground Truth

Computes PSA directly from experimental and near-experimental 3D coordinates for Cyclosporin A.  
Goal: establish the ground-truth ΔPSA that our vacuum Tier-1 pipeline (84.9 Å²) and future OpenMM MD should reproduce.

**Target**: Witek et al. 2016 (J. Chem. Inf. Model.) reported ΔPSA ≈ 75–80 Å² from 10 µs GROMOS MD + Markov State Models.

**Structures used**:
| Source | Environment | Type |
|---|---|---|
| PDB 1CYB | Cyclophilin-bound | NMR, 20 conformers (open/bound — not free drug) |
| CCDC 2149649 | Aqueous (A1 conformer) | Neutron+X-ray diffraction, JACS 2022 |
| Kessler 1985 / Loosli 1985 | CDCl₃ (closed) | Crystal / NMR (manual input if CIF unavailable) |
| CsA_start.xyz | Vacuum (RDKit ETKDGv3) | Our pipeline starting geometry |

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem, rdMolDescriptors
from rdkit.Chem import rdFreeSASA
RDLogger.DisableLog('rdApp.*')

print('RDKit ready')

## 1. Polar PSA Function (exact replication of conformer_engine.py)

Uses Bondi radii + rdFreeSASA, heavy-atom-only polar definition (N, O, S, P).  
All results in this notebook use this function so comparisons are apples-to-apples with the Tier-1 pipeline.

In [ ]:
# Exact copy of _polar_sasa() from conformer_engine.py
_BONDI = {
    'H': 1.20, 'C': 1.70, 'N': 1.55, 'O': 1.52,
    'S': 1.80, 'P': 1.80, 'F': 1.47, 'Cl': 1.75,
    'Br': 1.85, 'I': 1.98,
}
_POLAR_ELEMENTS = {'N', 'O', 'S', 'P'}

def compute_polar_sasa(mol, conf_id=0):
    """
    Compute polar SASA using Bondi radii + rdFreeSASA.
    Polar = N, O, S, P (heavy atoms only, no polar-H).
    Identical to conformer_engine.py _polar_sasa().
    Returns np.nan on failure.
    """
    try:
        radii = []
        for atom in mol.GetAtoms():
            sym = atom.GetSymbol()
            radii.append(_BONDI.get(sym, 1.50))
            if sym in _POLAR_ELEMENTS:
                atom.SetIntProp('SASAClass', 0)
                atom.SetProp('SASAClassName', 'Polar')
            else:
                atom.SetIntProp('SASAClass', 1)
                atom.SetProp('SASAClassName', 'APolar')
        query = rdFreeSASA.MakeFreeSasaPolarAtomQuery()
        psa = rdFreeSASA.CalcSASA(mol, radii, confIdx=conf_id, query=query)
        return round(float(psa), 4)
    except Exception as e:
        print(f'PSA error: {e}')
        return np.nan

print('PSA function loaded')

## 2. CsA Starting Geometry (Vacuum ETKDGv3)

Load CsA_start.xyz from the CREST run directory. This is the RDKit-generated starting geometry used as input for CREST (vacuum, single conformer). Compute PSA as baseline.

In [ ]:
import re

def xyz_to_rdkit_mol(xyz_path, smiles):
    """
    Load an XYZ file and assign 3D coordinates to an RDKit mol built from SMILES.
    Matches atoms by element order — assumes XYZ atom order matches SMILES heavy atoms + H.
    """
    # Parse XYZ
    with open(xyz_path) as f:
        lines = f.readlines()
    n_atoms = int(lines[0].strip())
    coords = []
    elements = []
    for line in lines[2:2+n_atoms]:
        parts = line.split()
        elements.append(parts[0])
        coords.append([float(parts[1]), float(parts[2]), float(parts[3])])

    # Build mol from SMILES with explicit H
    mol = Chem.MolFromSmiles(smiles)
    mol = Chem.AddHs(mol)

    if mol.GetNumAtoms() != n_atoms:
        print(f'WARNING: atom count mismatch: mol={mol.GetNumAtoms()}, xyz={n_atoms}')
        return None

    # Assign coordinates
    conf = Chem.Conformer(n_atoms)
    from rdkit.Geometry import rdGeometry
    for i, (x, y, z) in enumerate(coords):
        conf.SetAtomPosition(i, (x, y, z))
    mol.AddConformer(conf, assignId=True)
    return mol

# CsA SMILES (canonical, from CycPeptMPDB)
CSA_SMILES = 'CC[C@@H]1OC(=O)[C@H](CC(C)C)N(C)C(=O)[C@@H](CC(C)C)N(C)C(=O)[C@@H](CC(C)C)N(C)C(=O)[C@@H](Cc2ccccc2)N(C)C(=O)[C@H](C)NC(=O)[C@H](CC(C)C)N(C)C(=O)[C@@H](CC(C)C)NC(=O)[C@H](CC(C)C)N(C)C(=O)[C@H](C)N(C)C1=O'

XYZ_PATH = Path('/home/j4carmon/projects/CHEM_269_Final_Project/results/crest_runs/CsA/CsA_start.xyz')

csa_mol_xyz = xyz_to_rdkit_mol(XYZ_PATH, CSA_SMILES)
if csa_mol_xyz:
    psa_xyz = compute_polar_sasa(csa_mol_xyz, conf_id=0)
    print(f'CsA_start.xyz PSA: {psa_xyz:.2f} Å²')
    print(f'Atoms in mol: {csa_mol_xyz.GetNumAtoms()}')
else:
    print('SMILES/XYZ mismatch — trying alternative SMILES')
    # Fallback: generate conformer from SMILES directly
    mol = Chem.MolFromSmiles(CSA_SMILES)
    mol = Chem.AddHs(mol)
    AllChem.EmbedMolecule(mol, AllChem.ETKDGv3())
    AllChem.MMFFOptimizeMolecule(mol)
    psa_xyz = compute_polar_sasa(mol, conf_id=0)
    print(f'ETKDGv3 generated PSA: {psa_xyz:.2f} Å²')
    csa_mol_xyz = mol

## 3. Tier-1 Pipeline Results (from existing CSV)

Load aq_psa3d (max-PSA conformer) and mem_psa3d (min-PSA conformer) for CsA from the 7k conformer run.

In [ ]:
raw_csv = Path('/home/j4carmon/projects/CHEM_269_Final_Project/data/conformer_descriptors_raw_7k.csv')
fm_csv  = Path('/home/j4carmon/projects/Chameleon_Predictor/results/feature_matrix.csv')

raw = pd.read_csv(raw_csv)
fm  = pd.read_csv(fm_csv, low_memory=False)

# Find CsA by MW (~1203 Da) and name
csa_fm = fm[fm['MolWt'].between(1200, 1210)].copy()
print(f'CsA candidates by MW: {len(csa_fm)}')
print(csa_fm[['ID','MolWt','PAMPA','Source','Original_Name_in_Source_Literature']].to_string())

if len(csa_fm) > 0:
    csa_id = csa_fm['ID'].iloc[0]
    csa_raw = raw[raw['ID'] == csa_id]
    if len(csa_raw) > 0:
        r = csa_raw.iloc[0]
        print(f'\nTier-1 results for CsA (ID={csa_id}):')
        print(f'  aq_psa3d  (aqueous / max-PSA conformer): {r.aq_psa3d:.2f} Å²')
        print(f'  mem_psa3d (membrane / min-PSA conformer): {r.mem_psa3d:.2f} Å²')
        print(f'  delta_psa3d: {r.delta_psa3d:.2f} Å²')
        print(f'  n_confs used: {r.n_confs_used:.0f}')
        TIER1_AQ_PSA  = r.aq_psa3d
        TIER1_MEM_PSA = r.mem_psa3d
        TIER1_DELTA   = r.delta_psa3d
    else:
        print('CsA not found in raw conformer CSV')
        TIER1_AQ_PSA = TIER1_MEM_PSA = TIER1_DELTA = np.nan
else:
    print('CsA not found by MW in feature matrix')
    TIER1_AQ_PSA = TIER1_MEM_PSA = TIER1_DELTA = np.nan

## 4. PDB 1CYB — NMR Conformers (CsA bound to Cyclophilin)

20 NMR models of CsA bound to cyclophilin. **Caveat**: this is the open/bound conformation —  
not the free drug in either solvent. PSA here represents the protein-bound extended conformation,  
not the aqueous or membrane free conformation. Included for completeness and comparison.

In [ ]:
import urllib.request

pdb_path = Path('/tmp/1CYB.pdb')
if not pdb_path.exists():
    urllib.request.urlretrieve('https://files.rcsb.org/download/1CYB.pdb', pdb_path)
print(f'1CYB downloaded: {pdb_path.stat().st_size/1024:.1f} KB')

def extract_ligand_from_pdb(pdb_path, smiles, res_names=('CSA','CYC','CPH')):
    """
    Extract all MODEL blocks for a ligand from a PDB file.
    Returns list of RDKit mols (one per NMR model).
    """
    from rdkit.Chem import rdchem
    from rdkit.Geometry import rdGeometry

    with open(pdb_path) as f:
        content = f.read()

    # Split into MODEL blocks
    models = []
    current = []
    in_model = False
    for line in content.split('\n'):
        if line.startswith('MODEL'):
            in_model = True
            current = []
        elif line.startswith('ENDMDL'):
            models.append(current)
            in_model = False
        elif in_model:
            current.append(line)

    if not models:
        # Single model
        models = [content.split('\n')]

    print(f'Found {len(models)} MODEL blocks')

    results = []
    for model_lines in models:
        hetatm = [l for l in model_lines
                  if l.startswith('HETATM') and any(r in l for r in res_names)]
        if not hetatm:
            # Try ATOM records too
            hetatm = [l for l in model_lines
                      if l.startswith('ATOM') and any(r in l for r in res_names)]
        if not hetatm:
            continue

        # Parse xyz from PDB HETATM columns
        atoms_xyz = []
        for line in hetatm:
            try:
                elem = line[76:78].strip() or line[12:16].strip()[0]
                x, y, z = float(line[30:38]), float(line[38:46]), float(line[46:54])
                atoms_xyz.append((elem, x, y, z))
            except:
                pass
        results.append(atoms_xyz)

    return results

ligand_models = extract_ligand_from_pdb(pdb_path, CSA_SMILES)
print(f'Ligand models extracted: {len(ligand_models)}')
if ligand_models:
    print(f'Atoms per model: {len(ligand_models[0])}')

In [ ]:
# Build RDKit mol from SMILES and assign each model's coordinates as a conformer
def build_mol_from_pdb_models(ligand_models, smiles):
    """
    Attempt to assign PDB HETATM coordinates to an RDKit mol built from SMILES.
    Only uses heavy atoms (no H from PDB since PDB often omits H).
    Returns mol with one conformer per NMR model, or None if atom count mismatch.
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    mol = Chem.AddHs(mol)

    # Get heavy atom indices
    heavy_idx = [a.GetIdx() for a in mol.GetAtoms() if a.GetAtomicNum() != 1]

    psa_values = []
    for i, atoms_xyz in enumerate(ligand_models):
        heavy_pdb = [(e, x, y, z) for e, x, y, z in atoms_xyz if e != 'H']
        if len(heavy_pdb) != len(heavy_idx):
            continue

        # Build conformer with all atoms; assign heavy atoms from PDB, H at origin
        conf = Chem.Conformer(mol.GetNumAtoms())
        for j, (e, x, y, z) in enumerate(heavy_pdb):
            conf.SetAtomPosition(heavy_idx[j], (x, y, z))
        # Leave H positions at 0,0,0 — we only need polar-heavy PSA
        mol_copy = Chem.RWMol(mol)
        mol_copy.AddConformer(conf, assignId=True)

        psa = compute_polar_sasa(mol_copy, conf_id=mol_copy.GetNumConformers()-1)
        psa_values.append(psa)

    return psa_values

pdb_psas = build_mol_from_pdb_models(ligand_models, CSA_SMILES)

if pdb_psas:
    pdb_psas_clean = [p for p in pdb_psas if not np.isnan(p)]
    print(f'1CYB NMR PSA values ({len(pdb_psas_clean)} conformers):')
    print(f'  Mean: {np.mean(pdb_psas_clean):.2f} Å²')
    print(f'  Min:  {np.min(pdb_psas_clean):.2f} Å²')
    print(f'  Max:  {np.max(pdb_psas_clean):.2f} Å²')
    print(f'  Std:  {np.std(pdb_psas_clean):.2f} Å²')
    PDB_MEAN_PSA = np.mean(pdb_psas_clean)
else:
    print('Could not parse PDB conformer PSA values')
    PDB_MEAN_PSA = np.nan

## 5. CCDC 2149649 — Aqueous CsA (JACS 2022)

Neutron + X-ray diffraction structure of the A1 aqueous CsA conformer.  
This is the experimentally confirmed aqueous ground-truth structure.  

**Download instructions (manual)**:  
1. Go to https://www.ccdc.cam.ac.uk/structures/  
2. Search deposition number: 2149649  
3. Download CIF file → save to `/tmp/csa_aqueous_2149649.cif`  

Reference: Vithani et al., JACS 2022, DOI: 10.1021/jacs.2c01743

In [ ]:
cif_path = Path('/tmp/csa_aqueous_2149649.cif')

if cif_path.exists():
    print('CIF file found — parsing coordinates...')
    try:
        import gemmi
        doc = gemmi.cif.read(str(cif_path))
        block = doc.sole_block()
        # Extract fractional coordinates
        xs = block.find_values('_atom_site_fract_x')
        ys = block.find_values('_atom_site_fract_y')
        zs = block.find_values('_atom_site_fract_z')
        types = block.find_values('_atom_site_type_symbol')
        print(f'  Atoms in CIF: {len(list(xs))}')
        print('  gemmi parsed successfully — coordinate extraction needed')
        # TODO: convert fractional → Cartesian, assign to mol, compute PSA
        CCDC_AQ_PSA = np.nan  # placeholder until parsing implemented
    except ImportError:
        print('gemmi not installed. Run: pip install gemmi')
        CCDC_AQ_PSA = np.nan
    except Exception as e:
        print(f'CIF parse error: {e}')
        CCDC_AQ_PSA = np.nan
else:
    print('CIF not found at /tmp/csa_aqueous_2149649.cif')
    print('Manual download required from CCDC (see instructions above)')
    CCDC_AQ_PSA = np.nan
    print(f'CCDC_AQ_PSA set to: {CCDC_AQ_PSA} (pending download)')

## 6. CDCl₃ Reference — Kessler 1985 / Loosli 1985 Closed Conformer

The classic closed CsA conformer in apolar solvent features:
- 4 intramolecular H-bonds (backbone NH buried)
- Twisted β-pleated sheet with type II' β-turn
- Substantially lower PSA than the aqueous conformer

If CIF unavailable from CCDC: estimate from ETKDGv3 min-PSA conformer  
and Tier-1 pipeline mem_psa3d value.

In [ ]:
# Try CCDC 2149650 (CDCl3 structure, paired with 2149649)
cif_cdcl3 = Path('/tmp/csa_cdcl3_2149650.cif')

if cif_cdcl3.exists():
    print('CDCl3 CIF found — parsing...')
    CCDC_MEM_PSA = np.nan  # implement same as above
else:
    print('CDCl3 CIF not found at /tmp/csa_cdcl3_2149650.cif')
    print('Falling back to Tier-1 mem_psa3d as CDCl3 proxy')
    CCDC_MEM_PSA = np.nan

# Generate ETKDGv3 ensemble and get min-PSA as best available proxy
print('\nGenerating 50-conformer ETKDGv3 ensemble for min-PSA estimate...')
mol_smi = Chem.MolFromSmiles(CSA_SMILES)
mol_smi = Chem.AddHs(mol_smi)
params = AllChem.ETKDGv3()
params.useSmallRingTorsions = True
params.useBasicKnowledge = True
params.randomSeed = 42
params.numThreads = 0
AllChem.EmbedMultipleConfs(mol_smi, numConfs=50, params=params)
AllChem.MMFFOptimizeMoleculeConfs(mol_smi)

n_confs = mol_smi.GetNumConformers()
print(f'Generated {n_confs} conformers')

psa_all = [compute_polar_sasa(mol_smi, i) for i in range(n_confs)]
psa_all = [p for p in psa_all if not np.isnan(p)]

ETKDG50_MAX_PSA = max(psa_all)
ETKDG50_MIN_PSA = min(psa_all)
ETKDG50_MEAN_PSA = np.mean(psa_all)
ETKDG50_DELTA = ETKDG50_MAX_PSA - ETKDG50_MIN_PSA

print(f'\nETKDGv3 50-conformer ensemble:')
print(f'  Max PSA (aqueous proxy):  {ETKDG50_MAX_PSA:.2f} Å²')
print(f'  Min PSA (membrane proxy): {ETKDG50_MIN_PSA:.2f} Å²')
print(f'  Mean PSA:                 {ETKDG50_MEAN_PSA:.2f} Å²')
print(f'  ΔPSA (max-min):           {ETKDG50_DELTA:.2f} Å²')

## 7. Benchmark Summary Table

In [ ]:
import matplotlib.pyplot as plt

rows = [
    {
        'Source':      'Witek 2016 (GROMOS MD + MSM)',
        'Environment': 'Water / CHCl₃',
        'PSA_aq':      '~155–160 (est.)',
        'PSA_mem':     '~75–85 (est.)',
        'delta_PSA':   '~75–80',
        'Notes':       'Literature target. MD-derived, not directly downloadable.',
    },
    {
        'Source':      'Tier-1 ETKDGv3 20-conf (our pipeline)',
        'Environment': 'Vacuum (PSA-ranked)',
        'PSA_aq':      f'{TIER1_AQ_PSA:.2f}' if not np.isnan(TIER1_AQ_PSA) else 'N/A',
        'PSA_mem':     f'{TIER1_MEM_PSA:.2f}' if not np.isnan(TIER1_MEM_PSA) else 'N/A',
        'delta_PSA':   f'{TIER1_DELTA:.2f}' if not np.isnan(TIER1_DELTA) else '84.9 (reported)',
        'Notes':       'Max/min PSA selection. No solvation.',
    },
    {
        'Source':      'ETKDGv3 50-conf (this notebook)',
        'Environment': 'Vacuum (PSA-ranked)',
        'PSA_aq':      f'{ETKDG50_MAX_PSA:.2f}',
        'PSA_mem':     f'{ETKDG50_MIN_PSA:.2f}',
        'delta_PSA':   f'{ETKDG50_DELTA:.2f}',
        'Notes':       '50 conformers. More sampling, still vacuum.',
    },
    {
        'Source':      'PDB 1CYB (NMR, cyclophilin-bound)',
        'Environment': 'Protein-bound (open)',
        'PSA_aq':      f'{PDB_MEAN_PSA:.2f}' if not np.isnan(PDB_MEAN_PSA) else 'N/A',
        'PSA_mem':     'N/A',
        'delta_PSA':   'N/A',
        'Notes':       'Bound conformation, not free drug. Not comparable.',
    },
    {
        'Source':      'CCDC 2149649 (JACS 2022, neutron+X-ray)',
        'Environment': 'Aqueous (A1 conformer)',
        'PSA_aq':      f'{CCDC_AQ_PSA:.2f}' if not np.isnan(CCDC_AQ_PSA) else 'Pending download',
        'PSA_mem':     'N/A',
        'delta_PSA':   'N/A',
        'Notes':       'Experimental ground truth for aqueous PSA.',
    },
    {
        'Source':      'CCDC 2149650 (CDCl₃ crystal)',
        'Environment': 'CDCl₃ (closed conformer)',
        'PSA_aq':      'N/A',
        'PSA_mem':     f'{CCDC_MEM_PSA:.2f}' if not np.isnan(CCDC_MEM_PSA) else 'Pending download',
        'delta_PSA':   'Pending both CIFs',
        'Notes':       'Experimental ground truth for membrane PSA.',
    },
]

bench_df = pd.DataFrame(rows)
print('CsA ΔPSA Benchmark Summary')
print('=' * 80)
print(bench_df[['Source','Environment','PSA_aq','PSA_mem','delta_PSA','Notes']].to_string(index=False))

In [ ]:
# PSA distribution across the 50-conformer ensemble
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.hist(psa_all, bins=20, color='steelblue', edgecolor='white')
ax1.axvline(ETKDG50_MAX_PSA, color='dodgerblue', linestyle='--', label=f'Max (aq proxy): {ETKDG50_MAX_PSA:.1f} Å²')
ax1.axvline(ETKDG50_MIN_PSA, color='tomato', linestyle='--', label=f'Min (mem proxy): {ETKDG50_MIN_PSA:.1f} Å²')
ax1.axvline(ETKDG50_MEAN_PSA, color='orange', linestyle='-', label=f'Mean: {ETKDG50_MEAN_PSA:.1f} Å²')
ax1.set_xlabel('Polar SASA (Å²)')
ax1.set_ylabel('Count')
ax1.set_title('CsA: PSA distribution across\n50 vacuum ETKDGv3 conformers')
ax1.legend(fontsize=8)

# Delta PSA comparison bar chart
labels = ['Witek 2016\n(target)', 'Tier-1\n20-conf', 'This nb\n50-conf']
values = [77.5, TIER1_DELTA if not np.isnan(TIER1_DELTA) else 84.9, ETKDG50_DELTA]
colors = ['green', 'steelblue', 'steelblue']
bars = ax2.bar(labels, values, color=colors, alpha=0.8, edgecolor='white')
ax2.axhline(75, color='green', linestyle=':', alpha=0.5, label='Witek lower bound')
ax2.axhline(80, color='green', linestyle=':', alpha=0.5, label='Witek upper bound')
ax2.fill_between([-0.5, 2.5], 75, 80, alpha=0.1, color='green', label='Target range')
for bar, val in zip(bars, values):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{val:.1f}', ha='center', va='bottom', fontsize=10)
ax2.set_ylabel('ΔPSA (Å²)')
ax2.set_title('CsA ΔPSA: Our pipeline vs\nWitek 2016 target')
ax2.legend(fontsize=8)
ax2.set_ylim(0, 110)

plt.tight_layout()
plt.savefig('/tmp/csa_psa_benchmark.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved to /tmp/csa_psa_benchmark.png')

## 8. Interpretation and Next Steps

### What the benchmark tells us

**If ΔPSA (50-conf vacuum) ≈ ΔPSA (20-conf, Tier-1)**:  
Conformer count (20 vs 50) is not the limiting factor — the vacuum assumption is the bottleneck.

**If ΔPSA (vacuum) > Witek target (~75-80 Å²)**:  
Vacuum over-estimates the aqueous PSA (extended conformer in vacuum is more extended than in water),  
or under-estimates the membrane PSA (collapsed conformer not fully explored in vacuum).  
Both imply OpenMM GBSA-OBC MD will give a more accurate, lower ΔPSA.

**CCDC structures (once downloaded)**:  
- PSA(CCDC 2149649) = experimental aqueous PSA → target for `aq_psa3d`  
- PSA(CCDC 2149650) = experimental CDCl₃ PSA → target for `mem_psa3d`  
- If OpenMM MD recovers both within ~5 Å² → method validated for Furukawa scale-up

### OpenMM target

The OpenMM GBSA-OBC MD experiment should aim to recover:  
- Mean ensemble PSA(water) ≈ CCDC 2149649 value (once obtained)  
- Mean ensemble PSA(CHCl₃) ≈ CCDC 2149650 value (once obtained)  
- ΔPSA_ensemble ≈ 75-80 Å² (Witek target)

**CCDC download instructions (required to complete this benchmark)**:  
1. https://www.ccdc.cam.ac.uk/structures/ → search 2149649 → Download CIF  
2. https://www.ccdc.cam.ac.uk/structures/ → search 2149650 → Download CIF  
3. Save to `/tmp/csa_aqueous_2149649.cif` and `/tmp/csa_cdcl3_2149650.cif`  
4. Re-run Sections 5 and 6 above